In [1]:
import pandas as pd                   

file_path = "PHOENIX_TRAIN.csv"       # DataFrame structure: encodes a spreadsheet in Python. Two-dimensional data structure: rows and columns. Each column has a header and each row has an index
df = pd.read_csv(file_path)           # Creates a DataFrame variable 'df', using the read_csv command to load the "PHOENIX_TRAIN.csv" file from the root directory
df.head(15)

,de,dgs
0,und nun die wettervorhersage für morgen donner...,JETZT WETTER MORGEN DONNERSTAG ZWOELF FEBRUAR
1,mancherorts regnet es auch länger und ergiebig...,ORT REGEN DURCH REGEN KOENNEN UEBERSCHWEMMUNG ...
2,im nordwesten bleibt es heute nacht meist troc...,NORDWEST HEUTE NACHT TROCKEN BLEIBEN SUEDWEST ...
3,auch am tag gibt es verbreitet zum teil kräfti...,TAGSUEBER OFT REGEN GEWITTER KOENNEN MANCHMAL ...
4,größere wolkenlücken finden sich vor allem im ...,WOLKE LOCH SPEZIELL NORDWEST
5,im emsland heute nacht nur neun am oberrhein b...,FLUSS HEUTE NACHT SECHS FLUSS SIEBZEHN GRAD
6,morgen ähnliche temperaturen wie heute neunzeh...,MORGEN TEMPERATUR GLEICH-WIE HEUTE NEUNZEHN BI...
7,am freitag kann es in der osthälfte teilweise ...,FREITAG OST KOENNEN REGEN STARK REGEN VORSICHT...
8,sonst wechselhaft mit schauern und gewittern d...,REGION WECHSELHAFT REGEN GEWITTER AUCH WOCHENENDE
9,am temperaturniveau ändert sich wenig,TEMPERATUR BLEIBEN GLEICH


In [2]:
import sys

print("--- STARTING ENVIRONMENT CHECK ---")
print(f"Using Python version: {sys.version}\n")

# --- Checking Main Libraries ---

try:
    import transformers
    print(f"Transformers OK. Installed version: {transformers.__version__}")
except ImportError:
    print("ERROR: 'transformers' library not found.")

try:
    import torch
    print(f"PyTorch (torch) OK. Installed version: {torch.__version__}")
except ImportError:
    print("ERROR: 'torch' library not found.")

try:
    import datasets
    print(f"Datasets OK. Installed version: {datasets.__version__}")
except ImportError:
    print("ERROR: 'datasets' library not found.")

try:
    import accelerate
    print(f"Accelerate OK. Installed version: {accelerate.__version__}")
except ImportError:
    print("ERROR: 'accelerate' library not found.")

try:
    import pandas
    print(f"Pandas OK. Installed version: {pandas.__version__}")
except ImportError:
    print("ERROR: 'pandas' library not found.")

try:
    import sacrebleu
    print(f"Sacrebleu OK. Installed version: {sacrebleu.__version__}")
except ImportError:
    print("ERROR: 'sacrebleu' library not found.")

try:
    import evaluate
    print(f"Evaluate OK. Installed version: {evaluate.__version__}")
except ImportError:
    print("ERROR: 'evaluate' library not found.")


# --- Checking GPU Access ---
print("\n--- CHECKING GPU (CUDA) ACCESS ---")
try:
    import torch
    if torch.cuda.is_available():
        print(f"SUCCESS! PyTorch found the GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("WARNING: PyTorch is installed, but could NOT find your local GPU.")
except Exception as e:
    print(f"ERROR: Failed to check GPU status with PyTorch: {e}")

print("\n--- VERIFICATION COMPLETED ---")

--- STARTING ENVIRONMENT CHECK ---
Using Python version: 3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]

Transformers OK. Installed version: 4.57.1
PyTorch (torch) OK. Installed version: 2.14.1+cu130
Datasets OK. Installed version: 4.4.1
Accelerate OK. Installed version: 1.11.0
Pandas OK. Installed version: 2.3.3
Sacrebleu OK. Installed version: 2.2.1
Evaluate OK. Installed version: 0.4.0

--- CHECKING GPU (CUDA) ACCESS ---
SUCCESS! PyTorch found the GPU: NVIDIA GeForce RTX 5070 Ti

--- VERIFICATION COMPLETED ---


In [3]:
import os

# The script to be executed
generator_script = "gen_variants.py"

# Step 1: Check if the input files exist
# The "!" executes a shell command directly from the Jupyter Notebook and returns the output
print("Checking input files:")
!ls -l PHOENIX_*.csv # ls -> list; -l: long format; PHOENIX_*.csv matches any file starting with PHOENIX_ and ending with .csv

# --- Step 2: Execute the script ---
print("\nExecuting the gen_variants.py script...")
!python {generator_script}   # Calls the Python interpreter in the shell, passing the {generator_script} variable. The shell reads "python gen_variants.py"
print("\nScript execution completed!")

# --- Step 3: Verify the output ---
print("\nContents of the 'variants' folder (Should display 6 CSV files and 2 JSON/TXT files):")
!ls -l variants/ # Lists the files inside the variants folder

Checking input files:
-rw-r--r-- 1 j170860@rsrch.cepetro.unicamp.br domain users@rsrch.cepetro.unicamp.br  68465 fev  4  2026 PHOENIX_DEV.csv
-rw-r--r-- 1 j170860@rsrch.cepetro.unicamp.br domain users@rsrch.cepetro.unicamp.br  79353 fev  4  2026 PHOENIX_TEST.csv
-rw-r--r-- 1 j170860@rsrch.cepetro.unicamp.br domain users@rsrch.cepetro.unicamp.br 999880 fev  4  2026 PHOENIX_TRAIN.csv

Executing the gen_variants.py script...
--- STARTING DATA PROCESSING (PHOENIX DATASET) ---

--- Processing: TRAIN ---
Initial total: 7096

--- Processing: VAL ---
Initial total: 519

--- Processing: TEST ---
Initial total: 642

--- Saving Files ---
Saved: de-dgs.train.csv
Saved: de-dgs.val.csv
Saved: de-dgs.test.csv
Saved: de-dgs.lower.train.csv
Saved: de-dgs.lower.val.csv
Saved: de-dgs.lower.test.csv

--- Generating Vocabulary ---
Vocabulary saved. Total unique glosses: 1085
Process completed successfully!

Script execution completed!

Contents of the 'variants' folder (Should display 6 CSV files and 2 JSO

In [4]:
# ==============================================================================
# TRAINING WITH DATA AUGMENTATION - mBART - PHOENIX-14T
# ==============================================================================

import os                                   # Library to create folders and save files
import json
import random
import numpy as np                          # Main numerical library in Python
import torch                                # Python engine (torch and Dataloader handle raw GPU computations)
import pandas as pd                         # Main library for data manipulation and analysis
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm
from bitsandbytes.optim import AdamW8bit    # AdamW8bit optimizer compressed to 8 bits (formerly 32), allowing mBART training without blowing up memory

from transformers import (                  # Provides the tokenizer, LR scheduler (linear, cosine) 
    AutoTokenizer, 
    get_scheduler, 
    AutoModelForSeq2SeqLM, 
    DataCollatorForSeq2Seq, 
    AutoConfig
)
import datasets                             # Provides dataset handling structures
datasets.config.TORCHVISION_AVAILABLE = False
import sys
if "torchvision" in sys.modules:
    del sys.modules["torchvision"]
import evaluate                             # Provides evaluation metrics
from accelerate import Accelerator          # Manages data traffic between CPU RAM and GPU VRAM

# Creates text labels to avoid typing paths like "./variants" repeatedly
DATASET_PATH = "./variants"
dataset_variant = "lower"
MSRC_LANG = "de"
AUX_LANG = "dgs"
RUN = "SEED_999"                            # ATTENTION TO THE RUN NAME !!!!!!!!!!!!!!!!!!!!!!!!!!!!!

# Hyperparameters
BATCH_SIZE = 4       # Number of sentences processed simultaneously before calculating errors to update weights            
gradient_accumulation_steps = 4 # Instead of taking 16 sentences at once... takes 4, calculates error, but DOES NOT UPDATE. Then +4, +4, +4, and then updates weights. Saves memory while using an effective batch size of 16
N_EPOCHS = 200                                                                # ATTENTION TO THE NUMBER OF EPOCHS !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
LEARNING_RATE = 5e-5                                                          # ATTENTION TO THE LEARNING RATE    !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!                                       
LR_SCHEDULER = "linear"    # LR advances linearly to the final value
SRC_LANG = 'de'   
TGT_LANG = 'dgs'
MAX_INPUT_LENGTH = 256     # Limits input size to 256 tokens, saving memory
MAX_TGT_LENGTH = 256       # Limits target sequence size to 256 tokens as well

VARIANT = '.' + dataset_variant if dataset_variant != 'plain' else '' # VARIANT text label -> .lower or empty if no variant

### DATASETS 
TRAIN_PATH = os.path.join(DATASET_PATH, f"de-dgs.augmented.train.csv") # Training dataset path: "./variants/de-dgs.augmented.train.csv"
TEST_PATH = os.path.join(DATASET_PATH, f"de-dgs{VARIANT}.test.csv") # Test dataset path: ./variants/de-dgs.lower.test.csv
DEV_PATH = os.path.join(DATASET_PATH, f"de-dgs{VARIANT}.val.csv") 
VAR_NAME = dataset_variant
output_dir = f'runs/{MSRC_LANG}-{AUX_LANG}-{VAR_NAME}-run{RUN}' # Output directory and file name: runs/de-dgs-lower-runname
os.makedirs(output_dir, exist_ok=True) # Creates folder with this name in the directory. If it already exists, continues without error

# Hardware Configuration: FIXES SEEDS FOR RANDOM, NUMPY, CPU TORCH, AND GPU CUDA. THIS ENSURES EVERY RUN STARTS WITH THE SAME WEIGHTS
SEED = 999
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==============================================================================
# 1. POST-PROCESSING FUNCTION: "TRANSLATING THE TRANSLATION" (IDS -> SENTENCES)
# ==============================================================================
def postprocess(predictions, labels):
    predictions = predictions.cpu().numpy() # Transfers predictions (tensors) from GPU to CPU, then converts them to a NumPy array
    labels = labels.cpu().numpy()           # Does the same for the labels (ground-truth)
    decoded_preds = tokenizers['pt_X'].batch_decode(predictions, skip_special_tokens=True) # Transforms ID vector into word vector, discarding special tokens (e.g., </s>), keeping only words
    labels = np.where(labels != -100, labels, tokenizers['pt_X'].pad_token_id) # If the ID value is different from -100, keep it. Otherwise, replace with pad token ID
    decoded_labels = tokenizers['pt_X'].batch_decode(labels, skip_special_tokens=True) # Same ID to word transformation process, now for labels
    decoded_preds = [pred.strip() for pred in decoded_preds] # Removes extra whitespace from sentences. Ex: " I  BUY  CAR" -> "I BUY CAR".
    decoded_labels = [[label.strip()] for label in decoded_labels] # DOES THE SAME FOR LABELS
    return decoded_preds, decoded_labels # RETURNS DECODED PREDICTIONS AND LABELS (WORDS, NOT IDS)

# ==============================================================================
# 2. DATA PREPARATION: STEP WHERE DATA CHANGES FROM WORDS TO NUMBERS FOR MBART TO UNDERSTAND
# ==============================================================================
vocab_file = os.path.join(DATASET_PATH, 'gloss_vocab.txt') # Passes the address of the gloss vocabulary txt file: "./variants/gloss_vocab.txt". Using os.path.join prevents the code from breaking on Windows (\) vs Linux (/)
with open(vocab_file, 'r') as f:  # Opens the text file in read mode
    gloss_vocab = [line.strip() for line in f if line.strip()] # For loop to generate list: reads line by line stripping "\n", generating a list

# Loads mBART Tokenizer
MODEL_NAME = "facebook/mbart-large-50-many-to-many-mmt" 
print(f"Loading Tokenizer: {MODEL_NAME}")
tokenizers = {'pt_X': AutoTokenizer.from_pretrained(MODEL_NAME)} # Goes to the internet, downloads the mBART dictionary, and stores it in a Python dictionary under the key 'pt_X'

# Language configuration
tokenizers['pt_X'].src_lang = "de_DE" # Forces the source language to be German from Germany (de_DE)
tokenizers['pt_X'].tgt_lang = "de_DE" # Forces the output language to be German as well, despite being DGS. If we invented 'dgs_de', mBART wouldn't recognize it. By forcing German, we use its pre-learned vocabulary to generate the output
tokenizers['pt_X'].add_tokens(list(gloss_vocab)) # Adds the gloss list to the mBART vocabulary. Without this, if mBART read the gloss SCHWESTER (sister), it would split it into subwords (e.g., SCH + WES + TER). With this, the gloss becomes a single atomic token, essential for the error metric. 

print("Loading CSVs...") # Function that converts pandas dataframes to DatasetDict format, which is more efficient for training.
raw_datasets = datasets.DatasetDict({  
    'train': datasets.Dataset.from_pandas(pd.read_csv(TRAIN_PATH)),
    'validation': datasets.Dataset.from_pandas(pd.read_csv(DEV_PATH)),
    'test': datasets.Dataset.from_pandas(pd.read_csv(TEST_PATH))
})

raw_datasets = raw_datasets.filter(lambda ex: pd.notna(ex['de']) and pd.notna(ex['dgs']) and len(str(ex['de']).strip()) > 0) # Filters values, dropping rows where the German sentence is null, the gloss is null, or the line is empty.

# Function that transforms texts into numbers
def preprocess_function(batch):
    inputs = [str(text) for text in batch["de"]] # Extracts inputs and forces them to be strings, preventing numbers or NaNs from disrupting training
    targets = [str(text) for text in batch["dgs"]] # Same for ground-truths
    
    # Tokenization
    model_inputs = tokenizers['pt_X'](inputs, max_length=MAX_INPUT_LENGTH, truncation=True) # Converts German sentences (input) to number matrices, limiting to 256 tokens.
    labels = tokenizers['pt_X'](text_target=targets, max_length=MAX_TGT_LENGTH, truncation=True) # Converts targets to number matrices (IDs)
        
    model_inputs["labels"] = labels["input_ids"] # labels["input_ids"] -> represents the gloss IDs and is passed as the ground-truth for the model
    return model_inputs # Returns the package with input_ids in German, and gloss labels

print("Configuring Tokenization...")
tokenized_datasets = raw_datasets.map(   # Applies the preprocessing function described above to all three datasets (train, val, and test)
    preprocess_function, 
    batched=True,  # Sends blocks of 1000 values at a time
    remove_columns=raw_datasets["train"].column_names # After transforming everything into numbers, drops the raw texts (strings)
)
tokenized_datasets.set_format("torch") # Transforms everything into PyTorch tensors. The GPU only understands PyTorch tensors

# ==============================================================================
# 3. TRAINING PREPARATION: ESTABLISHING CHECKPOINTS & INITIALIZATION
# ==============================================================================
best_bleu = 0.0
checkpoint_config = os.path.join(output_dir, "config.json") # Defines the path to look for the "config.json" file, which confirms that training occurred and the file was saved. 
# runs/de-dgs-lower-runname/config.json

if os.path.exists(checkpoint_config):   # Checks if the config.json file already exists in the run folder; if so, loads the acquired weights
    print(f"\nCHECKPOINT FOUND AT: {output_dir}")
    # CRITICAL FIX: Ensure the model is loaded directly onto the GPU to prevent bitsandbytes state tensor mismatch
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir).to(DEVICE) # Loads the weights the model has already adjusted
    try:
        with open(os.path.join(output_dir, 'stats.json'), 'r') as f: # Opens stats.json (statistics file) in read mode
            old_stats = json.load(f)                                 # Converts from JSON to Python dictionary
            if 'bleu' in old_stats and len(old_stats['bleu']) > 0:   # Checks if bleu exists and is greater than 0
                best_bleu = max(old_stats['bleu'])                   # If so, stores the best BLEU 
            print(f"   -> History recovered. Previous best BLEU: {best_bleu:.2f}")
    except:
        pass # Ignores errors and continues code execution
else: # If the "config.json" file isn't in the run folder, assumes the model has never been trained and starts from scratch
    print(f"STARTING TRAINING FROM SCRATCH: {MODEL_NAME}")
    config = AutoConfig.from_pretrained(MODEL_NAME)   # Downloads the mathematical skeleton of mBART
    config.dropout = 0.3           # Required dropout                                                     # ATTENTION TO DROPOUT !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    config.attention_dropout = 0.2 # Extra protection for mBART attention matrices                        # ATTENTION TO DROPOUT !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    
    # CRITICAL FIX: Initialize model directly on the DEVICE (GPU) so AdamW8bit anchors its states properly
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, config=config).to(DEVICE) # Downloads parameters and enforces dropout settings
    
    model.resize_token_embeddings(len(tokenizers['pt_X'])) # Resizes the mBART embeddings matrix to support the new vocabulary (gloss_vocab). E.g., creates a new matrix (from 250,000x1024 to 253,000x1024) with the associated glosses 
    # The weights of these newly added glosses are random, while existing words are preserved

# ==============================================================================
# OPTIMIZER WITH DIFFERENTIAL LRs
# ==============================================================================
# 1. Isolating Embedding layer weights
embedding_params = model.get_input_embeddings().parameters() # Stores the weights of the mBART Embedding layers
embedding_param_ids = {id(p) for p in embedding_params}      # Stores the memory addresses of all Embedding weights

# 2. Stores the rest of the mBART body (which is already pre-trained and needs protection)
body_params = [p for p in model.parameters() if id(p) not in embedding_param_ids] # Parameters are all those not associated with the embedding memory addresses

# 3. Defining different LRs. The body uses a conservative LR and embeddings use an LR x times larger.
optimizer_grouped_parameters = [ # Applies different LRs to the embedding layers and the body
    {"params": body_params, "lr": LEARNING_RATE},                            # Ex: 5e-5     # ATTENTION TO DIFFERENT LR RATES!!!!!!!!!!!!!!!!!
    {"params": model.get_input_embeddings().parameters(), "lr": LEARNING_RATE * 10}, # Ex: 2.5e-4
]

optimizer = AdamW8bit(optimizer_grouped_parameters, weight_decay=0.05) # Initializes the 8-bit AdamW optimizer with adjusted optimization parameters and WD=0.05

data_collator = DataCollatorForSeq2Seq(tokenizers['pt_X'], model=model) # Instead of forcing everything to size 256, evaluates the largest item in the batch and pads the smaller ones with <pad>. Ex: batch with sentences of 4, 5, 6, and 10 words. Pads smaller words with <pad> so all share the same size (10). 

train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, collate_fn=data_collator, batch_size=BATCH_SIZE)
eval_dataloader = DataLoader(tokenized_datasets["validation"], collate_fn=data_collator, batch_size=BATCH_SIZE)

accelerator = Accelerator(gradient_accumulation_steps=gradient_accumulation_steps)
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

# Exact calculation considering the optimizer only steps every 'gradient_accumulation_steps'
num_training_steps = (len(train_dataloader) // gradient_accumulation_steps) * N_EPOCHS

# 5% warm-up is ideal for 14k sentences
warmup_steps = int(num_training_steps * 0.05)              # ATTENTION TO WARM-UP

lr_scheduler = get_scheduler(
    LR_SCHEDULER, 
    optimizer=optimizer, 
    num_warmup_steps=warmup_steps, 
    num_training_steps=num_training_steps
)
metric = evaluate.load("sacrebleu")

# Forced language ID for mBART generation
forced_bos_token_id = tokenizers['pt_X'].lang_code_to_id["de_DE"]

# ==============================================================================
# 4. TRAINING LOOP
# ==============================================================================
patience = 10                                                                         # ATTENTION TO PATIENCE!!!!!!!!!!!!!!!!!!!!!!!!!
patience_counter = 0

print(f"\nSTARTING TRAINING | Epochs: {N_EPOCHS} | Best BLEU to beat: {best_bleu:.2f}")
progress_bar = tqdm(range(num_training_steps))
writer = SummaryWriter(log_dir=f'{output_dir}/logs')

stats = {'bleu': []}

for epoch in range(N_EPOCHS):
    # --- TRAINING ---
    model.train()
    total_loss = 0
    
    loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100)          # ATTENTION TO LABEL SMOOTHING!!!!!!!!!!!!!!!!!!!!!
    
    for batch in train_dataloader:
        outputs = model(**batch)
        
        logits = outputs.logits
        labels = batch["labels"]
        
        # OPTIMIZED LOSS CALCULATION FOR MEMORY 
        loss = loss_fct(logits.transpose(1, 2), labels)
        
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss)
        
        if (progress_bar.n + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            
        total_loss += loss.item() * gradient_accumulation_steps
        progress_bar.update(1)
    
    avg_loss = total_loss / len(train_dataloader)
    writer.add_scalar('Training/Loss', avg_loss, epoch)

    # --- VALIDATION ---
    model.eval()
    print(f"\n--- Epoch {epoch+1} ---")   
    for batch in tqdm(eval_dataloader, desc="Validation", leave=False):
        with torch.no_grad():
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"], 
                attention_mask=batch["attention_mask"], 
                max_length=MAX_TGT_LENGTH,
                forced_bos_token_id=forced_bos_token_id,
                num_beams=5,                                                                         # ATTENTION TO NUM_BEAMS
            )
        
        labels = batch["labels"]
        generated_tokens = accelerator.pad_across_processes(generated_tokens, dim=1, pad_index=tokenizers['pt_X'].pad_token_id)
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)
        
        predictions_gathered = accelerator.gather(generated_tokens)
        labels_gathered = accelerator.gather(labels)
        decoded_preds, decoded_labels = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=decoded_preds, references=decoded_labels)

    # PURIST EVALUATION
    results = metric.compute(tokenize='none', lowercase=True)
    current_bleu = results['score']
    
    print(f" Loss: {avg_loss:.4f} | Validation BLEU: {current_bleu:.2f} (Best: {best_bleu:.2f})")
    writer.add_scalar('BLEU/Validation', current_bleu, epoch)
    stats['bleu'].append(current_bleu)

    if current_bleu > best_bleu:
        print(f"New Record! (Old: {best_bleu:.2f} -> New: {current_bleu:.2f}) Saving model...")
        best_bleu = current_bleu
        patience_counter = 0
        accelerator.wait_for_everyone()
        accelerator.unwrap_model(model).save_pretrained(output_dir, save_function=accelerator.save)
        tokenizers['pt_X'].save_pretrained(output_dir)
        
        with open(os.path.join(output_dir, 'stats.json'), 'w') as f:
            json.dump(stats, f)
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
        
    if patience_counter >= patience:
        print("Early stopping. Model is no longer learning.")
        break

writer.close()
print("Training completed.")

Loading Tokenizer: facebook/mbart-large-50-many-to-many-mmt
Loading CSVs...


Filter:   0%|          | 0/14192 [00:00<?, ? examples/s]

Filter:   0%|          | 0/519 [00:00<?, ? examples/s]

Filter:   0%|          | 0/642 [00:00<?, ? examples/s]

Configuring Tokenization...


Map:   0%|          | 0/14192 [00:00<?, ? examples/s]

... (16 outputs left out)

Validation:   0%|          | 0/130 [00:00<?, ?it/s]

 Loss: 2.6011 | Validation BLEU: 26.48 (Best: 24.59)
New Record! (Old: 24.59 -> New: 26.48) Saving model...

--- Epoch 5 ---


Validation:   0%|          | 0/130 [00:00<?, ?it/s]

 Loss: 2.4867 | Validation BLEU: 25.72 (Best: 26.48)
No improvement. Patience: 1/1
Early stopping. Model is no longer learning.
Training completed.


In [ ]:
##################################################################################

In [ ]:
##################################################################################

In [5]:
# ==============================================================================
# TRAINING SCRIPT - PHASE 2 (EXPERT FINE-TUNING) mBART PHOENIX-14T
# ==============================================================================

import os                                   # Interacts with the operating system (create folders, manage paths)
import json                                 # Handles JSON files (used here to save the best BLEU history)
import random                               # Python's native random number generator
import numpy as np                          # Standard library for linear algebra calculations and matrices
import torch                                # The main mathematical engine executing Deep Learning on the GPU
import pandas as pd                         # Manipulates CSV spreadsheets with training and testing data
from torch.utils.data import DataLoader     # Packages data into batches to send to the GPU
from torch.utils.tensorboard import SummaryWriter # Creates real-time loss and BLEU charts to track training
from tqdm.auto import tqdm                  # Creates visual progress bars in the terminal
from bitsandbytes.optim import AdamW8bit    # Advanced optimizer running in 8 bits to save VRAM

from transformers import (                  # Imports Hugging Face tools
    AutoTokenizer,                          # Loads the "dictionary" that converts text into numbers
    get_scheduler,                          # Controls the learning rate rhythm (e.g., warm-up and decay)
    AutoModelForSeq2SeqLM,                  # Loads the model's Encoder-Decoder architecture
    DataCollatorForSeq2Seq,                 # Pads short sentences with <pad> so all have the same length in the batch
    AutoConfig                              # Loads the model's blueprint to modify the dropout
)
import datasets                             # Optimized library for handling large volumes of text
from accelerate import Accelerator          # The orchestrator for fluid communication between CPU, GPU, and memory
import evaluate                             # Official library to calculate translation metrics (SacreBLEU)

# --- 1. PATH CONFIGURATION ---
DATASET_PATH = "./variants"                 # Folder where original and augmented CSVs are saved
dataset_variant = "lower"                   # Indicates we are using lowercase texts
MSRC_LANG = "de"                            # Source language code (German)
AUX_LANG = "dgs"                            # Target language code (German Sign Language Gloss)

# ATTENTION HERE: Indicates which folder the model should load the Phase 1 (Generalist) weights from
PREV_RUN_DIR = f'runs/{MSRC_LANG}-{AUX_LANG}-{dataset_variant}-runSEED_999' # ATTENTION HERE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# Defines the new folder name where Phase 2 will be saved
RUN = "SEED_999_FINAL"                     # ATTENTION HERE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!                                

# --- PHASE 2 HYPERPARAMETERS ---
BATCH_SIZE = 4                              # Number of sentences processed simultaneously per iteration
gradient_accumulation_steps = 4             # Accumulates gradients for 4 steps before updating weights (effective batch = 16)
N_EPOCHS = 40                               # Reduced number of epochs (40) because this is just a fine refinement (expert fine-tuning)
LEARNING_RATE = 1e-5                        # Learning rate much lower than Phase 1 (5e-5) to avoid catastrophic forgetting
LR_SCHEDULER = "linear"                     # Learning rate rises during warm-up then decays linearly
WEIGHT_DECAY = 0.1                          # More aggressive penalty on large weights to prevent data memorization (overfitting)
SRC_LANG = 'de'                             # Key for the German column in the CSV
TGT_LANG = 'dgs'                            # Key for the gloss column in the CSV
MAX_INPUT_LENGTH = 256                      # Truncates input sentences longer than 256 tokens
MAX_TGT_LENGTH = 256                        # Truncates output sentences longer than 256 tokens

VARIANT = '.' + dataset_variant if dataset_variant != 'plain' else '' 
# In Phase 2, we load the ORIGINAL dataset (.train.csv), without artificially augmented data, so the model becomes an "expert" on clean data
TRAIN_PATH = os.path.join(DATASET_PATH, f"de-dgs{VARIANT}.train.csv") 
TEST_PATH = os.path.join(DATASET_PATH, f"de-dgs{VARIANT}.test.csv")
DEV_PATH = os.path.join(DATASET_PATH, f"de-dgs{VARIANT}.val.csv") 

output_dir = f'runs/{MSRC_LANG}-{AUX_LANG}-{dataset_variant}-run{RUN}' # Creates the final path string
os.makedirs(output_dir, exist_ok=True)      # Effectively creates the folder; if it already exists, proceeds without error

# Freezes random seeds for all libraries to guarantee exact experimental reproducibility
SEED = 999
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') # Checks for GPU availability and allocates there

# ==============================================================================
# 2. AUXILIARY FUNCTIONS AND DATA PREPARATION
# ==============================================================================
print(f"Loading Tokenizer from pre-trained model: {PREV_RUN_DIR}")
# Loads the exact tokenizer (already containing the added gloss vocabulary) from Phase 1
tokenizers = {'pt_X': AutoTokenizer.from_pretrained(PREV_RUN_DIR)}

# Strict mBART rule: We force both input and output to use the German grammatical flow
tokenizers['pt_X'].src_lang = "de_DE"
tokenizers['pt_X'].tgt_lang = "de_DE"
# Saves the exact ID number that tells the decoder: "Start speaking in German!"
forced_bos_token_id = tokenizers['pt_X'].lang_code_to_id["de_DE"]

print("Loading ORIGINAL CSVs (No Augmentation)...")
raw_datasets = datasets.DatasetDict({       # Reads the three CSVs using Pandas and packages them into the ultra-fast 'datasets' format
    'train': datasets.Dataset.from_pandas(pd.read_csv(TRAIN_PATH)),
    'validation': datasets.Dataset.from_pandas(pd.read_csv(DEV_PATH)),
    'test': datasets.Dataset.from_pandas(pd.read_csv(TEST_PATH))
})

# Scans the data and discards any empty rows or rows containing null values (NaN) to prevent crashes
raw_datasets = raw_datasets.filter(lambda ex: pd.notna(ex['de']) and pd.notna(ex['dgs']) and len(str(ex['de']).strip()) > 0)

# Standardizes column names internally to facilitate manipulation
raw_datasets = raw_datasets.map(
    lambda ex: {"translation": {SRC_LANG: str(ex[SRC_LANG]), TGT_LANG: str(ex[TGT_LANG])}},
    remove_columns=[SRC_LANG, TGT_LANG]
)

def preprocess_function(examples):
    # Separates the German sentences (input) and glosses (output)
    inputs = [str(ex[SRC_LANG]) for ex in examples["translation"]]
    targets = [str(ex[TGT_LANG]) for ex in examples["translation"]]
    
    # Clean mBART logic: Converts text into matrices of IDs (numbers), truncating excess
    model_inputs = tokenizers['pt_X'](inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizers['pt_X'](text_target=targets, max_length=MAX_TGT_LENGTH, truncation=True)
    
    # Attaches the gloss ID matrix inside the main package as the "labels" key (the ground-truth)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing...")
tokenized_datasets = raw_datasets.map(      # Applies the function above to all datasets in blocks (batched=True) for much faster processing
    preprocess_function, 
    batched=True,
    remove_columns=raw_datasets["train"].column_names # Deletes the original string text from RAM, keeping only the tensors
)
tokenized_datasets.set_format("torch")      # Converts normal numerical lists into PyTorch Tensors (the only language the GPU understands)

# ==============================================================================
# 3. MODEL & OPTIMIZER
# ==============================================================================
print(f"\nLoading Model: {PREV_RUN_DIR}")
config = AutoConfig.from_pretrained(PREV_RUN_DIR) # Pulls the model's skeleton
config.dropout = 0.2                              # Reduces dropout in the Expert Phase, as there is less risk of the model destabilizing with noisy data
model = AutoModelForSeq2SeqLM.from_pretrained(PREV_RUN_DIR, config=config) # Loads the "brains" (adjusted weights) from Phase 1

# --- DIFFERENTIAL LEARNING RATES (DLR) ---
# 1. Grabs weights only from the vocabulary layer (Embeddings) and stores their memory addresses
embedding_params = model.get_input_embeddings().parameters()
embedding_param_ids = {id(p) for p in embedding_params}
    
# 2. Grabs the rest of the neural network (Body) excluding the embedding memory locations
body_params = [p for p in model.parameters() if id(p) not in embedding_param_ids]
    
# 3. Creates distinct rules: the body updates very slowly (1e-5), but the vocabulary layer updates 10 times faster 
optimizer_grouped_parameters = [
    {"params": body_params, "lr": LEARNING_RATE},                            
    {"params": model.get_input_embeddings().parameters(), "lr": LEARNING_RATE * 10}, # ATTENTION HERE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
]

# Initializes 8-bit AdamW, injecting the differential rates calculated above and weight_decay (which punishes excessively large weights)
optimizer = AdamW8bit(optimizer_grouped_parameters, weight_decay=WEIGHT_DECAY)

# The Data Collator observes the largest sentence size in the mini-batch and fills (pads) empty spaces in other sentences with zeros
data_collator = DataCollatorForSeq2Seq(tokenizers['pt_X'], model=model)

# Batch iterator constructors: send data from RAM to GPU sliced by BATCH_SIZE (4)
train_dataloader = DataLoader(tokenized_datasets["train"], shuffle=True, collate_fn=data_collator, batch_size=BATCH_SIZE)
eval_dataloader = DataLoader(tokenized_datasets["validation"], collate_fn=data_collator, batch_size=BATCH_SIZE)

# Initializes the Accelerator, which handles math accounting for gradient accumulation and precision management
accelerator = Accelerator(gradient_accumulation_steps=gradient_accumulation_steps)
# The prepare method "injects" Accelerator superpowers into the original objects
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)

# Calculates how many actual gradient descent math steps we will take in total
actual_training_steps = (len(train_dataloader) // gradient_accumulation_steps) * N_EPOCHS

# 10% WARM-UP: During the first 10% of steps, the learning rate goes from zero to the maximum stipulated value, avoiding model shocks
warmup_steps = int(actual_training_steps * 0.10)

# Creates the temporal manager for the Learning Rate curve
lr_scheduler = get_scheduler(
    LR_SCHEDULER, 
    optimizer=optimizer, 
    num_warmup_steps=warmup_steps, 
    num_training_steps=actual_training_steps
)

metric = evaluate.load("sacrebleu")         # Instantiates the official translation metric

def postprocess(predictions, labels):
    # Function to clean sentences generated by the model so the SacreBLEU metric can evaluate them humanely
    predictions = predictions.cpu().numpy() # Brings matrices from the GPU back to the Processor
    labels = labels.cpu().numpy()
    # Converts numbers back to words
    decoded_preds = tokenizers['pt_X'].batch_decode(predictions, skip_special_tokens=True)
    # Ignores "pads" (filled spaces) by swapping -100 values for the neutral ID
    labels = np.where(labels != -100, labels, tokenizers['pt_X'].pad_token_id)
    decoded_labels = tokenizers['pt_X'].batch_decode(labels, skip_special_tokens=True)
    # Removes residual spaces at the edges of sentences
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [[label.strip()] for label in decoded_labels]
    return decoded_preds, decoded_labels

# ==============================================================================
# 4. TRAINING LOOP (PHASE 2)
# ==============================================================================
best_bleu = 0.0                             # We start with a zeroed BLEU record
patience = 10                               # Tolerance: If BLEU doesn't improve after 10 consecutive epochs, we stop everything (Early Stopping)
patience_counter = 0                        # Counter for failed epochs

print(f"\nSTART PHASE 2 | Epochs: {N_EPOCHS} | LR: {LEARNING_RATE}")
progress_bar = tqdm(range(actual_training_steps)) # Visual progress bar
writer = SummaryWriter(log_dir=f'{output_dir}/logs') # Log system for TensorBoard

stats = {'bleu': []}                        # Statistics dictionary to save history

for epoch in range(N_EPOCHS):
    model.train()                           # Turns on "knowledge absorption" mode (enables gradients and dropout)
    total_loss = 0
    
    # Label Smoothing of 0.1 means the model receives the instruction: "Never be absolutely 100% sure of the word; distribute 10% of your certainty among other words". Prevents Overfitting.
    loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100)
    
    for batch in train_dataloader:          # Starts pulling sentence batches from memory
        outputs = model(**batch)            # Forward pass: the entire network "tries" to translate the inputs
        logits = outputs.logits             # The model's raw mathematical guess
        labels = batch["labels"]            # The official ground-truth
        
        # Manual optimized calculation to avoid blowing up RAM
        loss = loss_fct(logits.transpose(1, 2), labels) 
        
        # Divides the accumulated error (loss) by 4 before sending it backward
        loss = loss / gradient_accumulation_steps
        accelerator.backward(loss)          # Backward pass: derivative math finds where the network erred
        
        # If we gathered a batch of 4, we actually update the model weights
        if (progress_bar.n + 1) % gradient_accumulation_steps == 0:
            optimizer.step()                # Takes a step descending the gradient mountain
            lr_scheduler.step()             # Adjusts the current step rhythm (step size)
            optimizer.zero_grad()           # Zeroes out the board's gradient memory for the next batch
            
        total_loss += loss.item() * gradient_accumulation_steps # Sums the statistical error to display at the end
        progress_bar.update(1)              # Moves one block on the visual progress bar
    
    avg_loss = total_loss / len(train_dataloader) # Calculates the average error for the entire epoch
    writer.add_scalar('Training/Loss', avg_loss, epoch) # Logs the error in the chart

    model.eval()                            # Locks knowledge absorption: now the network will only "take the test" for Validation
    print(f"\n--- Epoch {epoch+1} ---")
    
    for batch in tqdm(eval_dataloader, desc="Validation", leave=False):
        with torch.no_grad():               # Turns off gradient math entirely; we are just testing, which saves a lot of GPU
            # Commands the network to decode the full sentence without seeing the ground-truth. Beam=5 evaluates 5 possible grammatical "routes" and chooses the best.
            generated_tokens = accelerator.unwrap_model(model).generate(
                batch["input_ids"], 
                attention_mask=batch["attention_mask"], 
                max_length=MAX_TGT_LENGTH,
                forced_bos_token_id=forced_bos_token_id,
                num_beams=5
            )
        
        labels = batch["labels"]
        # If running on multiple GPUs (Multi-GPU DDP), pads the edges to align matrices across boards
        generated_tokens = accelerator.pad_across_processes(generated_tokens, dim=1, pad_index=tokenizers['pt_X'].pad_token_id)
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)
        
        # Pulls the result from all GPUs (if there are several) back to the main machine
        predictions_gathered = accelerator.gather(generated_tokens)
        labels_gathered = accelerator.gather(labels)
        # Cleans numbers into readable text words
        decoded_preds, decoded_labels = postprocess(predictions_gathered, labels_gathered)
        metric.add_batch(predictions=decoded_preds, references=decoded_labels) # Stores text in metric memory

    # Validation batch finished; now we ask to calculate the final test metric using strict standards (tokenize='none')
    results = metric.compute(tokenize='none', lowercase=True)
    current_bleu = results['score']
    
    print(f"Loss: {avg_loss:.4f} | Validation BLEU: {current_bleu:.2f} (Best: {best_bleu:.2f})")
    writer.add_scalar('BLEU/Validation', current_bleu, epoch)
    stats['bleu'].append(current_bleu)

    # Saving and Early Stopping Logic
    if current_bleu > best_bleu:            # Did the model break its own BLEU record?
        print(f"New Record! Saving model ...")
        best_bleu = current_bleu            # Updates the record
        patience_counter = 0                # Zeroes accumulated failures
        accelerator.wait_for_everyone()     # Synchronizes processes if in a Cluster (cluster wait barrier)
        accelerator.unwrap_model(model).save_pretrained(output_dir, save_function=accelerator.save) # Saves architecture in the folder
        tokenizers['pt_X'].save_pretrained(output_dir) # Saves dictionary in the folder
        
        with open(os.path.join(output_dir, 'stats.json'), 'w') as f: # Saves score in JSON for future auditing
            json.dump(stats, f)
    else:                                   # Didn't break the record? Takes a penalty point in patience
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{patience}")
        
    if patience_counter >= patience:        # If it bursts 10 consecutive failures, shuts everything down before epoch 40 arrives
        print("Early stopping. Model has reached its limit.")
        break

writer.close()                              # Turns off the graphing system to avoid leaking RAM
print("Phase 2 training completed. Proceed to the inference cell.")

Loading Tokenizer from pre-trained model: runs/de-dgs-lower-runSEED_999
Loading ORIGINAL CSVs (No Augmentation)...


Filter:   0%|          | 0/7096 [00:00<?, ? examples/s]

Filter:   0%|          | 0/519 [00:00<?, ? examples/s]

Filter:   0%|          | 0/642 [00:00<?, ? examples/s]

Map:   0%|          | 0/7096 [00:00<?, ? examples/s]

Map:   0%|          | 0/519 [00:00<?, ? examples/s]

... (17 outputs left out)

Loss: 2.2754 | Validation BLEU: 27.31 (Best: 27.69)
No improvement. Patience: 4/10

--- Epoch 6 ---


Validation:   0%|          | 0/130 [00:00<?, ?it/s]

Loss: 2.2257 | Validation BLEU: 25.49 (Best: 27.69)
No improvement. Patience: 5/10


KeyboardInterrupt: 

In [6]:
# ==============================================================================
# EVALUATION CELL: BLEU-1, BLEU-2, BLEU-3, BLEU-4, and ROUGE-L (mBART)
# ==============================================================================
import torch                                    # Math engine responsible for running matrices on the GPU
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # Loads the trained mBART architecture and dictionary
from tqdm.auto import tqdm                      # Creates the visual progress bar in the terminal
import pandas as pd                             # Library to manipulate and save CSV spreadsheets with results
import numpy as np                              # Library for numerical operations
import sacrebleu                                # OFFICIAL BLEU metric calculation library, required in high-impact papers
from sacrebleu.metrics import BLEU              # Specific module to dissect BLEU into exact n-grams (1, 2, 3, and 4)
import evaluate                                 # Hugging Face library to load standardized metrics (used here for ROUGE)
import os                                       # Manipulates operating system files and directories

# --- 1. CONFIGURATIONS ---
DATASET_PATH = "./variants"                     # Base folder where the original/variant CSVs are stored
dataset_variant = "lower"                       # Indicates the dataset variant (in this case, lowercase texts)
MSRC_LANG = "de"                                # Source language (German)
AUX_LANG = "dgs"                                # Target language (DGS Gloss)

# Defines from which training folder we will extract the model for the final test
RUN = "SEED_999"                           ############### ATTENTION TO THE NAME!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
SRC_LANG = 'de'                                 # Input column name in the CSV
TGT_LANG = 'dgs'                                # Target column name (ground-truth) in the CSV
MAX_INPUT_LENGTH = 256                          # The maximum token limit to process on input
MAX_TGT_LENGTH = 256                            # The maximum token limit allowed in the generated translation

# Since the testing phase does not calculate gradients (loss.backward()), the entire GPU VRAM 
# remains free exclusively for direct processing (forward pass), ensuring maximum speed.
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
VAR_NAME = dataset_variant

# Dynamic Paths
VARIANT = '.' + dataset_variant if dataset_variant != 'plain' else '' 
output_dir = f'runs/{MSRC_LANG}-{AUX_LANG}-{VAR_NAME}-run{RUN}'


# ATTENTION: To generate the final results for the paper's table, you must ensure 
# that the path below points to the ".test.csv" file, which is the blind set.
TEST_PATH = os.path.join(DATASET_PATH, f"de-dgs{VARIANT}.val.csv") # ATTENTION HERE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

print(f"Loading model from: {output_dir}")
print(f"Using test dataset: {TEST_PATH}")

# --- 2. LOAD MODEL AND DATA ---
try:
    # Loads the tokenizer and the adjusted weights (brain) of mBART directly from your training folder
    tokenizer = AutoTokenizer.from_pretrained(output_dir)
    model = AutoModelForSeq2SeqLM.from_pretrained(output_dir).to(DEVICE)
except OSError:
    print("ERROR: Model folder not found. Check the 'RUN' variable.")
    raise

# MANDATORY MBART CONFIGURATION
# mBART needs to explicitly know in which "grammatical corridor" it must start producing text
tokenizer.src_lang = "de_DE"
forced_bos_token_id = tokenizer.lang_code_to_id["de_DE"] # Forces the first word of the translation to be the German identifier

# Reads the test CSV
df_test = pd.read_csv(TEST_PATH)
# A safety measure: clears any row where German or the gloss is empty (NaN)
df_test = df_test[df_test['de'].notna() & df_test['dgs'].notna()]

# --- 3. GENERATION (INFERENCE) ---
print("\nGenerating translations (This may take a few minutes)...")
preds = []                                      # Empty list to store the machine's translations
refs = []                                       # Empty list to store the official ground-truths (original glosses)

# Locks the model in deterministic test mode. This turns off Dropout. 
# It is crucial, because during testing we don't want the network to randomly "forget" or modify information.
model.eval()

# Loop that iterates through each sentence in the test CSV, one by one
for index, row in tqdm(df_test.iterrows(), total=len(df_test)):
    input_text = str(row[SRC_LANG])             # Gets the German sentence
    target_text = str(row[TGT_LANG])            # Gets the original gloss (ground-truth)
    
    # Tokenizes the input sentence and sends the numbers to the GPU (pt = PyTorch format)
    inputs = tokenizer(input_text, return_tensors="pt", max_length=MAX_INPUT_LENGTH, truncation=True).to(DEVICE)
    
    with torch.no_grad():                       # Disables the autograd engine (saves 50% VRAM)
       # The model generates the output sentence
       outputs = model.generate(
            **inputs, 
            max_length=MAX_TGT_LENGTH,          # Up to 256 tokens max
            forced_bos_token_id=forced_bos_token_id, # Injects the "de_DE" tag at the start
            num_beams=5,                        # Tree search: evaluates the 5 most likely linguistic paths before deciding the final sentence
            #length_penalty=0.8,                # Penalty for short sentences
            #no_repeat_ngram_size=3,            # Would forbid the model from repeating the same trio of words in a row
            early_stopping=True                 # If the sentence hits the <end of sentence> token, it immediately stops processing beams
        )
        
    # Converts the generated numbers back into clean letters and words
    pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    preds.append(pred_text)                     # Saves the model's attempt
    refs.append(target_text)                    # Saves the ground-truth

# --- 4. METRICS CALCULATION ---
print("\nCalculating purist metrics...")

# The SacreBLEU library requires the references (ground-truths) format to be a "list of lists".
# This allows comparing 1 prediction against N possible correct ground-truths.
refs_transposed = [refs] 
# ROUGE, on the other hand, requires simple (flat) lists
refs_flat = refs

results = {}

# A) SacreBLEU (Unified official paper metric)
# The tokenize='none' argument is MANDATORY in sign language scientific papers. 
# If you leave the default tokenizer on, it splits words internally before counting (inflating scores). 
# With 'none', we force it to count the exact generated spaces, reflecting real rigor.
sb = sacrebleu.corpus_bleu(preds, refs_transposed, lowercase=True, tokenize='none')
results['SacreBLEU'] = sb.score

# B) BLEU 1, 2, 3, 4 
# Here we dissect the BLEU. BLEU-1 measures isolated word hits (unigrams).
# BLEU-4 measures if the model managed to generate exact blocks of 4 consecutive words from the ground-truth (difficult, measures deep syntax).
results['BLEU-1'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=1).corpus_score(preds, refs_transposed).score
results['BLEU-2'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=2).corpus_score(preds, refs_transposed).score
results['BLEU-3'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=3).corpus_score(preds, refs_transposed).score
results['BLEU-4'] = BLEU(lowercase=True, tokenize='none', max_ngram_order=4).corpus_score(preds, refs_transposed).score

# C) ROUGE (Recall-Oriented)
try:
    # ROUGE-L focuses on the longest common sequence between what was generated and the ground-truth. 
    # It is an essential metric because it captures sentence fluency, not just mechanical repetition.
    rouge = evaluate.load("rouge")
    rouge_res = rouge.compute(predictions=preds, references=refs_flat)
    results['ROUGE-L'] = rouge_res['rougeL'] * 100 # We multiply by 100 to match the BLEU scale (0-100)
except Exception as e:
    print(f"WARNING: Could not calculate ROUGE ({e})")
    results['ROUGE-L'] = 0.0

# --- 5. RESULTS DISPLAY ---
print("\n" + "="*40)
print(f"FINAL PERFORMANCE REPORT (mBART)")
print("="*40)
print(f"{'Metric':<15} | {'Score':<10}")
print("-" * 28)
print(f"{'BLEU-1 ':<15} | {results['BLEU-1']:.2f}")
print(f"{'BLEU-2 ':<15} | {results['BLEU-2']:.2f}")
print(f"{'BLEU-3 ':<15} | {results['BLEU-3']:.2f}")
print(f"{'BLEU-4 ':<15} | {results['BLEU-4']:.2f}")
print("-" * 28)
print(f"{'SacreBLEU':<15} | {results['SacreBLEU']:.2f}")
print(f"{'ROUGE-L':<15} | {results['ROUGE-L']:.2f}")
print("="*40)

# Finally, creates a spreadsheet (CSV) placing the German Sentence, Ground-truth (Gloss), and Model Attempt side-by-side.
# This spreadsheet is what you will use to select visual examples for the paper (the qualitative section).
df_result_file = pd.DataFrame({'de': df_test['de'], 'dgs_ref': refs, 'dgs_pred': preds})
save_file = os.path.join(output_dir, 'final_metrics_evaluation.csv')
df_result_file.to_csv(save_file, index=False)
print(f"\nDetailed file with translations saved at: {save_file}")

Loading model from: runs/de-dgs-lower-runSEED_999
Using test dataset: ./variants/de-dgs.lower.val.csv

Generating translations (This may take a few minutes)...


  0%|          | 0/519 [00:00<?, ?it/s]


Calculating purist metrics...

FINAL PERFORMANCE REPORT (mBART)
Metric          | Score     
----------------------------
BLEU-1          | 63.60
BLEU-2          | 46.26
BLEU-3          | 34.51
BLEU-4          | 26.59
----------------------------
SacreBLEU       | 26.59
ROUGE-L         | 62.98

Detailed file with translations saved at: runs/de-dgs-lower-runSEED_999/final_metrics_evaluation.csv
